# ⚡ Smart Home Energy Consumption — EDA & Modeling Pipeline
**Dataset Source:** [Kaggle: mexwell/smart-home-energy-consumption](https://www.kaggle.com/datasets/mexwell/smart-home-energy-consumption)

This notebook explores 100,000 records of smart home appliance energy telemetry across 500 households, outdoor temperature variations, seasons, and household sizes.

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error

# 1. Load Kaggle Dataset
data_path = os.path.join('..', '..', 'dataset', 'smart_home_energy_consumption_large.csv')
df = pd.read_csv(data_path)
print(f'Total records: {len(df):,}')
df.head()

## 2. Feature Engineering & Exploration
We extract `hour` from the 24-hour timestamp, `month`, and `is_weekend` from `Date`.

In [ ]:
df['hour'] = pd.to_datetime(df['Time'], format='%H:%M').dt.hour
df['date_dt'] = pd.to_datetime(df['Date'])
df['month'] = df['date_dt'].dt.month
df['is_weekend'] = (df['date_dt'].dt.dayofweek >= 5).astype(int)

# Appliance-level Consumption Benchmarks
appliance_summary = df.groupby('Appliance Type')['Energy Consumption (kWh)'].agg(
    ['count', 'mean', 'std', 'min', 'max']
).rename(columns={'mean': 'Mean (kWh)', 'count': 'Samples'})
appliance_summary.sort_values(by='Mean (kWh)', ascending=False)

## 3. Seasonal & Temperature Impact
Analyze average energy usage across seasons and ambient temperatures.

In [ ]:
season_summary = df.groupby('Season')['Energy Consumption (kWh)'].mean()
print('Average Energy Consumption by Season (kWh):')
print(season_summary)

# Household Size comparison
hh_summary = df.groupby('Household Size')['Energy Consumption (kWh)'].mean()
print('\nAverage Energy Consumption by Household Size:')
print(hh_summary)

## 4. Train/Test Split & Pipeline Modeling

In [ ]:
categorical_cols = ['Appliance Type', 'Season']
numerical_cols = ['Outdoor Temperature (°C)', 'Household Size', 'hour', 'month', 'is_weekend']
features = categorical_cols + numerical_cols
target = 'Energy Consumption (kWh)'

X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), categorical_cols),
    ('num', 'passthrough', numerical_cols)
])

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', HistGradientBoostingRegressor(max_iter=150, random_state=42))
])

pipeline.fit(X_train, y_train)
predictions = pipeline.predict(X_test)

print(f'R² Score: {r2_score(y_test, predictions):.4f}')
print(f'MAE:      {mean_absolute_error(y_test, predictions):.4f} kWh')
print(f'RMSE:     {root_mean_squared_error(y_test, predictions):.4f} kWh')